In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import shuffle
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import StratifiedShuffleSplit


In [3]:


def split_n_plus_1(X, y, n, random_state=42):
    X, y = shuffle(X, y, random_state=random_state)
    X_splits = np.array_split(X, n + 1)
    y_splits = np.array_split(y, n + 1)
    return X_splits, y_splits


In [4]:

def train_ensemble_lrs(X_splits, y_splits, n):
    lrs = []
    for i in range(n):
        lr = LogisticRegression(
            solver='saga',
           # penalty='l2',
            max_iter=2000,
            n_jobs=-1,
            verbose=1
        )
        
        print(f"Training lr {i + 1}!")
        lr.fit(X_splits[i], y_splits[i])
        print(f"Done lr {i + 1}!")
        lrs.append(lr)
    return lrs


In [5]:
def get_probability_features(models, X):
    """
    Output: (num_samples, n_models)
    """
    probs = []
    for model in models:
        p = model.predict_proba(X)[:, 1]  # lấy P(class=1)
        probs.append(p)

    return np.stack(probs, axis=1)


In [6]:
def train_meta_lr(Z, y):
    meta_lr = LogisticRegression(
        solver='saga',
        penalty='l2',
        max_iter=2000,
        n_jobs=-1,
        verbose=1
    )
    meta_lr.fit(Z, y)
    return meta_lr


In [7]:
def predict_stack(models, meta_model, X):
    Z = get_probability_features(models, X)
    y_pred = meta_model.predict(Z)
    y_prob = meta_model.predict_proba(Z)[:, 1]
    return y_pred, y_prob


In [ ]:


# ================== LOAD DATA ==================
X1 = np.load("D:/Pythonfile/Voice-Activity-Detect/data/feature/train/lfcc_non-speech_1.npy")
y1 = np.zeros(len(X1), dtype=int)

X2 = np.load("D:/Pythonfile/Voice-Activity-Detect/data/feature/train/lfcc_non-speech_2.npy")
y2 = np.zeros(len(X2), dtype=int)

X3 = np.load("D:/Pythonfile/Voice-Activity-Detect/data/feature/train/lfcc_speech.npy")
y3 = np.ones(len(X3), dtype=int)

print(X1.shape, X2.shape, X3.shape)


X12 = np.concatenate((X1, X2), axis=0)
y12 = np.concatenate((y1, y2), axis=0)
# idx = np.r_[:13, 27:39]
X = np.concatenate((X12, X3), axis=0)[:, :13]
y = np.concatenate((y12, y3), axis=0)
X, y = shuffle(X, y, random_state=42)

sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.9,   # 90% test, 10% train
    random_state=42
)

for train_idx, test_idx in sss.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

print(f"Train size: {len(X_train)}")
print(f"Test  size: {len(X_test)}")

n = 5  

X_splits, y_splits = split_n_plus_1(X_train, y_train, n)
ensemble_lrs = train_ensemble_lrs(X_splits, y_splits, n)

X_meta = X_splits[n]
y_meta = y_splits[n]

Z_meta = get_probability_features(ensemble_lrs, X_meta)
print("Meta feature shape:", Z_meta.shape)

final_lr = train_meta_lr(Z_meta, y_meta)

y_pred, y_prob = predict_stack(ensemble_lrs, final_lr, X_test)

acc = accuracy_score(y_test, y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {pre:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc:.4f}")


(10944158, 60) (1688068, 60) (12845018, 60)
Train size: 2547724
Test  size: 22929520
Training lr 1!


d:\Pythonfile\Voice-Activity-Detect\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
